# Capstone — Structured Content Archetype Clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainDev04/Flyrank-ML-Internship/blob/main/work/notebooks/capstone.ipynb)

**Lane 3.** This notebook mirrors the deployed research paper section by section and regenerates every
number it quotes, so a reader can check the paper against running code rather than against my memory.

📄 **Paper:** https://zaindev04.github.io/Flyrank-ML-Internship/
📦 **Repository:** https://github.com/ZainDev04/Flyrank-ML-Internship

> Skills: `skills/writing-research-papers/SKILL.md` + `skills/deploying-static-pages/SKILL.md`.
> Data: FlyRank ML Internship dataset — [flyrank.ai](https://flyrank.ai). Seed 42 throughout.

**The one-line version:** six behavioural archetypes reproduce across seeds; **five survive a
feature-removal test and one does not**; portability to unseen clients is directional (mean ARI 0.909,
worst 0.465); the deliverable is a ranked review queue with an explicit low-confidence bucket and no
automation.

In [1]:
import os, sys, json, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

SEED = 42
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ZainDev04/Flyrank-ML-Internship"
CSV_REL  = "data/raw/content_refresh_anonymized.csv"
if IN_COLAB and not Path(CSV_REL).exists():
    if not os.path.isdir("Flyrank-ML-Internship"):
        subprocess.run(["git","clone","--depth","1",REPO_URL,"Flyrank-ML-Internship"], check=True)
    os.chdir("Flyrank-ML-Internship")
else:
    here = Path.cwd()
    for c in [here, *here.parents]:
        if (c / CSV_REL).exists():
            os.chdir(c); break
assert Path(CSV_REL).exists()
pd.set_option("display.width", 190); pd.set_option("display.max_columns", 40)
print(f"sklearn {sklearn.__version__} | pandas {pd.__version__} | seed {SEED}")

sklearn 1.8.0 | pandas 3.0.2 | seed 42


## 1. Question

> **What behavioural archetypes exist across a content inventory, and what treatment does each deserve?**

Not *"which page is worst?"* — that is a ranking question and a different lane. A content lead with 18,000
pages sets a handful of policies and applies them; the useful unit is the **kind of page**, not the page.

**The tension the paper resolves:** a hand-written five-rung ladder does this job reasonably well, and puts
43% of the inventory into one catch-all bucket with a single recommendation between 8,130 pages.

In [2]:
MIN_IMPRESSIONS, K = 300, 6      # frozen since ML-02
df = pd.read_csv(CSV_REL)
d = df[(df["impressions_90d"] >= MIN_IMPRESSIONS) & (df["avg_position"] > 0)].copy()
d["log_impressions"] = np.log1p(d["impressions_90d"])
d["log_clicks"] = np.log1p(d["clicks_90d"])
d["log_sessions"] = np.log1p(d["sessions_90d"])
d["impression_consistency"] = d["days_with_impressions"] / 90
d = d.reset_index(drop=True)

def ladder(r):
    if r["impressions_90d"] >= 3000 and r["avg_position"] <= 10:          return "CHAMPION"
    if r["days_since_last_update"] >= 90 and r["impressions_90d"] >= 500: return "STALE_VISIBLE"
    if r["engagement_rate"] >= 10:                                        return "ENGAGED_NICHE"
    if r["days_with_impressions"] < 45:                                   return "INTERMITTENT"
    return "STEADY_LOW"
d["rule_archetype"] = d.apply(ladder, axis=1)

print(f"eligible inventory : {len(d):,} pages / {d['client_id'].nunique()} clients")
print(f"excluded           : {len(df)-len(d):,} pages "
      f"(median {df[~df['content_id'].isin(d['content_id'])]['impressions_90d'].median():.0f} impressions)")
print(f"\nthe hand-written ladder:")
print(d["rule_archetype"].value_counts().to_string())
big = d["rule_archetype"].value_counts()
print(f"\n'{big.idxmax()}' holds {big.max():,} pages = {big.max()/len(d)*100:.0f}% of the inventory,")
print("with one recommendation between all of them. That bucket is the paper's subject.")

eligible inventory : 18,752 pages / 29 clients
excluded           : 11,248 pages (median 31 impressions)

the hand-written ladder:
rule_archetype
STEADY_LOW       8130
STALE_VISIBLE    4891
CHAMPION         4571
ENGAGED_NICHE     903
INTERMITTENT      257

'STEADY_LOW' holds 8,130 pages = 43% of the inventory,
with one recommendation between all of them. That bucket is the paper's subject.


## 2. Data

**Two releases, different jobs.**

| Release | Used for | Scale |
|---|---|---|
| `data/raw/content_refresh_anonymized.csv` | the archetype analysis | 30,000 × 44, 32 clients |
| `FlyRank/internship-warehouse` (`month=2026-03`) | validating the data contract, windows, availability | 9,841,378 March rows; 78.8M total |

**Grain:** one row = one pseudonymized page, one client, one trailing 90-day window.

**Exclusions and why** — reproduced live below. Every identifier is a pseudonym; no client name, domain,
URL, title or raw query exists in the release or in the paper.

In [3]:
removed = df[~df["content_id"].isin(d["content_id"])]
print("EXCLUSIONS\n")
print(f"  impressions_90d < {MIN_IMPRESSIONS} or no position data : {len(removed):,} pages "
      f"({len(removed)/len(df)*100:.0f}%)")
print(f"    their median impressions_90d : {removed['impressions_90d'].median():.0f}")
print(f"    kept pages median            : {d['impressions_90d'].median():.0f}")
print(f"  avg_position == 0 ('no data', not rank zero) : {int((df['avg_position']==0).sum()):,}")
print(f"\n  GA4 engagement columns: excluded from the FEATURE set entirely.")
print("    ML-04 measured only 4.2% of warehouse March rows pass ga4_data_available IS TRUE.")
print("    In clustering, filler zeros do not blur a boundary - they draw one.\n")

assert d["content_id"].is_unique, "grain broken"
print(f"Grain check passed: {d['content_id'].nunique():,} unique ids in {len(d):,} rows.")
UNSAFE = ["url","domain","title","query","client_name","email"]
print(f"Public-safety: columns hinting at raw private text: "
      f"{[c for c in df.columns if any(h in c.lower() for h in UNSAFE)] or 'none'}")
print(f"Id format: {d['content_id'].iloc[0]} | {d['client_id'].iloc[0]}")

EXCLUSIONS

  impressions_90d < 300 or no position data : 11,248 pages (37%)
    their median impressions_90d : 31
    kept pages median            : 2399
  avg_position == 0 ('no data', not rank zero) : 1,205

  GA4 engagement columns: excluded from the FEATURE set entirely.
    ML-04 measured only 4.2% of warehouse March rows pass ga4_data_available IS TRUE.
    In clustering, filler zeros do not blur a boundary - they draw one.

Grain check passed: 18,752 unique ids in 18,752 rows.
Public-safety: columns hinting at raw private text: none
Id format: content_304f48230142 | client_f369cb89fc


## 3. Methodology

**Task type:** clustering, unsupervised. No observed outcome, no ground truth. "Archetype" is a construct
imposed to make policy tractable — stated because it governs every claim.

**Features (9, all standardised):** log impressions, log clicks, CTR, average position, log sessions,
engagement rate, impression consistency, content age, days since last update.

**Held out for validation, never used as features:** word count, character count, search volume,
competition, CPC, content type, main intent.

**Success criteria, fixed before results** (ML-03): stability ARI ≥ 0.80, silhouette > 0.15 reported
honestly, ≥ 3 held-out variables separating at p < 0.001, and a human sense-check with no duplicate
actions.

In [4]:
FEATURES = ["log_impressions","log_clicks","ctr","avg_position","log_sessions",
            "engagement_rate","impression_consistency","content_age_days","days_since_last_update"]
HELD_OUT = ["word_count","char_count","search_volume","competition","cpc"]
assert not set(FEATURES) & set(HELD_OUT), "a validator leaked into the feature set"

BANNED = ["trend_pct","trend_direction","client_id","content_id",
          "health_score","priority_score","action_type","rule_archetype","cluster"]
assert not set(FEATURES) & set(BANNED), "a banned column is in the feature set"
print(f"features: {len(FEATURES)}  |  held out for validation: {len(HELD_OUT)}")
print(f"banned from features: {', '.join(BANNED)}")
print("Guards passed.\n")

scaler = StandardScaler().fit(d[FEATURES].fillna(d[FEATURES].median()))
X = scaler.transform(d[FEATURES].fillna(d[FEATURES].median()))
km = KMeans(n_clusters=K, n_init=25, random_state=SEED).fit(X)
d["cluster"] = km.labels_
d["silhouette"] = silhouette_samples(X, km.labels_)
samp = np.random.RandomState(0).choice(len(X), 5000, replace=False)
print(f"K-Means k={K}: sizes {sorted(pd.Series(km.labels_).value_counts().tolist())}")

features: 9  |  held out for validation: 5
banned from features: trend_pct, trend_direction, client_id, content_id, health_score, priority_score, action_type, rule_archetype, cluster
Guards passed.

K-Means k=6: sizes [537, 985, 3595, 4113, 4598, 4924]


## 4. Results (vs the baseline)

In [5]:
gm = GaussianMixture(n_components=K, random_state=SEED, n_init=5).fit_predict(X)
print("METHOD CHOICE - identical features, identical scaling, identical k\n")
print(f"  {'K-Means':20s} silhouette {silhouette_score(X[samp], km.labels_[samp]):.3f}")
print(f"  {'Gaussian Mixture':20s} silhouette {silhouette_score(X[samp], gm[samp]):.3f}")
print("  -> the more expressive model scored half. Complexity earned nothing.\n")

ari_ladder = adjusted_rand_score(d["cluster"], d["rule_archetype"])
ladder_codes = pd.factorize(d["rule_archetype"])[0]
print(f"AGREEMENT with the hand-written ladder : ARI {ari_ladder:.3f}")
print(f"  silhouette of the ladder's partition : "
      f"{silhouette_score(X[samp], ladder_codes[samp]):.3f}")
print(f"  silhouette of the learned partition  : "
      f"{silhouette_score(X[samp], km.labels_[samp]):.3f}")
print("  The ladder is not incoherent - its edges are policy thresholds rather than the")
print("  data's own density, so it cuts across regions the data keeps together.")

METHOD CHOICE - identical features, identical scaling, identical k

  K-Means              silhouette 0.221
  Gaussian Mixture     silhouette 0.116
  -> the more expressive model scored half. Complexity earned nothing.

AGREEMENT with the hand-written ladder : ARI 0.288
  silhouette of the ladder's partition : 0.120
  silhouette of the learned partition  : 0.221
  The ladder is not incoherent - its edges are policy thresholds rather than the
  data's own density, so it cuts across regions the data keeps together.


In [6]:
steady = d[d["rule_archetype"] == "STEADY_LOW"]
inside = steady.groupby("cluster").agg(
    pages=("content_id","size"), median_impressions=("impressions_90d","median"),
    median_ctr=("ctr","median"), median_position=("avg_position","median"),
    median_age_days=("content_age_days","median")).sort_values("pages", ascending=False)
print(f"THE HEADLINE: opening the ladder's {len(steady):,}-page catch-all\n")
print(inside.round(2).to_string())
print("\nTwo findings in that table:")
print("  1. A ~123-day cohort and a ~445-day cohort at nearly identical volume and position")
print("     are the same row to the ladder - a 3.6x age difference no rung asked about.")
hi = inside.nlargest(1, "median_impressions")
print(f"  2. {int(hi['pages'].iloc[0]):,} pages carrying a median "
      f"{hi['median_impressions'].iloc[0]:,.0f} impressions at {hi['median_ctr'].iloc[0]:.2f}% CTR")
print("     were labelled 'steady low' - they failed the champion rung only on position.")

THE HEADLINE: opening the ladder's 8,130-page catch-all

         pages  median_impressions  median_ctr  median_position  median_age_days
cluster                                                                         
4         3365              1339.0        0.15             13.4            123.0
0         2920              1200.0        0.09             17.8            445.0
2          735             10620.0        0.42             14.4            147.0
3          588               430.0        0.00             12.8            144.0
1          522               399.0        0.00             17.4            284.0

Two findings in that table:
  1. A ~123-day cohort and a ~445-day cohort at nearly identical volume and position
     are the same row to the ladder - a 3.6x age difference no rung asked about.
  2. 735 pages carrying a median 10,620 impressions at 0.42% CTR
     were labelled 'steady low' - they failed the champion rung only on position.


In [7]:
print("VALIDATION - three tests, ascending order of honesty\n")
seed_ari = [adjusted_rand_score(km.labels_, KMeans(n_clusters=K, n_init=25,
            random_state=s).fit_predict(X)) for s in (1,7,13,99,2024)]
print(f"  1. seed stability (5 seeds)        : mean ARI {np.mean(seed_ari):.3f}")

sub_ari = []
for s in range(10):
    idx = np.random.RandomState(s).choice(len(X), int(0.8*len(X)), replace=False)
    sub_ari.append(adjusted_rand_score(km.labels_[idx],
                   KMeans(n_clusters=K, n_init=25, random_state=SEED).fit_predict(X[idx])))
print(f"  2. subsample stability (10 x 80%)  : mean ARI {np.mean(sub_ari):.3f} "
      f"(std {np.std(sub_ari):.3f})")

sizes = d["client_id"].value_counts()
loo = []
for c in sizes[sizes >= 100].index:
    tr, te = d[d["client_id"] != c], d[d["client_id"] == c]
    kmt = KMeans(n_clusters=K, n_init=25, random_state=SEED).fit(
        scaler.transform(tr[FEATURES].fillna(d[FEATURES].median())))
    loo.append(adjusted_rand_score(km.labels_[te.index],
               kmt.predict(scaler.transform(te[FEATURES].fillna(d[FEATURES].median())))))
loo = np.array(loo)
print(f"  3. LEAVE-ONE-CLIENT-OUT ({len(loo)} clients) : mean ARI {loo.mean():.3f} "
      f"(std {loo.std():.3f}, worst {loo.min():.3f})")
print(f"     clients below the 0.80 bar set in ML-03: {int((loo<0.80).sum())} of {len(loo)}")
print()
print(f"  The mean drops {np.mean(sub_ari)-loo.mean():.3f}. The SPREAD grows "
      f"{loo.std()/np.std(sub_ari):.0f}x. The spread is the finding:")
print("  a random subsample cannot fail, because every client sits on both sides of it.")

VALIDATION - three tests, ascending order of honesty

  1. seed stability (5 seeds)        : mean ARI 0.998
  2. subsample stability (10 x 80%)  : mean ARI 0.990 (std 0.004)
  3. LEAVE-ONE-CLIENT-OUT (17 clients) : mean ARI 0.909 (std 0.138, worst 0.465)
     clients below the 0.80 bar set in ML-03: 3 of 17

  The mean drops 0.081. The SPREAD grows 32x. The spread is the finding:
  a random subsample cannot fail, because every client sits on both sides of it.


## 5. Limitations

Written before a reader can write them for me.

In [8]:
REDUCED = [c for c in FEATURES if c != "engagement_rate"]
Xr = StandardScaler().fit_transform(d[REDUCED].fillna(d[REDUCED].median()))
lab_r = KMeans(n_clusters=K, n_init=25, random_state=SEED).fit_predict(Xr)
d["cluster_reduced"] = lab_r

print("THE ROBUSTNESS TEST THAT COST ME AN ARCHETYPE\n")
print(f"  silhouette with engagement_rate    : {silhouette_score(X[samp], km.labels_[samp]):.3f}")
print(f"  silhouette without                 : {silhouette_score(Xr[samp], lab_r[samp]):.3f}")
print(f"  ARI(full, reduced)                 : {adjusted_rand_score(km.labels_, lab_r):.3f}")
eng_c = d.groupby("cluster")["engagement_rate"].median().idxmax()
sub = d[d["cluster"] == eng_c]
dest = sub["cluster_reduced"].value_counts()
print(f"\n  the engagement group ({len(sub)} pages, median "
      f"{sub['engagement_rate'].median():.0f}% engagement) scatters:")
print(f"    largest destination holds {dest.iloc[0]/len(sub)*100:.0f}% of them")
print(f"    reduced-run medians: "
      f"{d.groupby('cluster_reduced')['engagement_rate'].median().round(2).tolist()}")
print("    -> no reduced cluster isolates engagement. WITHDRAWN as a feature artefact.\n")

pca = PCA(n_components=4).fit(X)
low = (d["silhouette"] < 0.05).mean()*100
print(f"OTHER LIMITS, measured:")
print(f"  silhouette {silhouette_score(X[samp], km.labels_[samp]):.3f} - weak separation, "
      "regions of a continuum")
print(f"  PC1 carries {pca.explained_variance_ratio_[0]*100:.0f}% and is a volume axis")
print(f"  {low:.1f}% of pages sit within 0.05 of a boundary")
print(f"  worst leave-one-client-out ARI: {loo.min():.3f}")
print(f"  ARI(K-Means, Gaussian Mixture) = {adjusted_rand_score(km.labels_, gm):.3f} - a different")
print("    method finds a substantially different partition of the same data")

THE ROBUSTNESS TEST THAT COST ME AN ARCHETYPE

  silhouette with engagement_rate    : 0.221
  silhouette without                 : 0.233
  ARI(full, reduced)                 : 0.783

  the engagement group (537 pages, median 25% engagement) scatters:
    largest destination holds 34% of them
    reduced-run medians: [0.0, 0.0, 0.0, 0.0, 2.56, 0.0]
    -> no reduced cluster isolates engagement. WITHDRAWN as a feature artefact.

OTHER LIMITS, measured:
  silhouette 0.221 - weak separation, regions of a continuum
  PC1 carries 34% and is a volume axis
  11.2% of pages sit within 0.05 of a boundary
  worst leave-one-client-out ARI: 0.465
  ARI(K-Means, Gaussian Mixture) = 0.288 - a different
    method finds a substantially different partition of the same data


**What this study cannot claim**, stated plainly and carried into the paper verbatim:

- **Not natural kinds.** k = 6 is a choice from a nearly flat silhouette curve. A different method finds a
  substantially different partition (ARI 0.288 vs Gaussian Mixture).
- **Not semantic clustering.** The release contains no article text. These are behavioural metrics and
  content metadata; calling it semantic would describe an analysis that was not done.
- **Not causal, not predictive.** One 90-day window, no intervention, no control group.
- **No claim about any search engine's ranking system.**
- **Portability is directional.** 14 of 17 clients above 0.80, three below, worst 0.465.
- **Two archetypes are ~half a single client each** (53% and 38% top-client share) — house styles common
  enough to form a group, not universal content types.

### The negative result the paper reports

The published FlyRank paper's most dramatic finding is that 365+ day content refreshed within 30 days shows
57× more impressions. I tested that comparison's shape on my slice.

In [9]:
old = df[df["content_age_days"] >= 365].copy()
old["refreshed_30d"] = old["days_since_last_update"] <= 30
g = old.groupby("refreshed_30d").agg(pages=("content_id","size"),
                                     median_impressions=("impressions_90d","median"))
ratio_full = g.loc[True,"median_impressions"] / max(g.loc[False,"median_impressions"], 1)
vis = old[(old["impressions_90d"] >= MIN_IMPRESSIONS) & (old["avg_position"] > 0)]
g2 = vis.groupby("refreshed_30d").agg(pages=("content_id","size"),
                                      median_impressions=("impressions_90d","median"))
ratio_vis = g2.loc[True,"median_impressions"] / max(g2.loc[False,"median_impressions"], 1)

print("365+ day content: recently refreshed vs not\n")
print("full slice:"); print(g.round(1).to_string())
print(f"  median impression ratio: {ratio_full:.2f}x")
print("\ninside my eligible population:"); print(g2.round(1).to_string())
print(f"  median impression ratio: {ratio_vis:.2f}x")
print(f"\n  Could not reproduce the DIRECTION of the published finding on this slice.")
print(f"  NOT evidence the paper is wrong: different dataset, and the decisive detail is")
print(f"  population - that paper's un-refreshed baseline sits at 71 median impressions,")
print(f"  and my filter removes {len(removed):,} pages whose median is "
      f"{removed['impressions_90d'].median():.0f}.")
print(f"  Honest conclusion, narrow: refresh recency is not a priority signal HERE.")

365+ day content: recently refreshed vs not

full slice:
               pages  median_impressions
refreshed_30d                           
False            553              1159.0
True            5807               817.0
  median impression ratio: 0.70x

inside my eligible population:
               pages  median_impressions
refreshed_30d                           
False            449              1642.0
True            4026              1748.5
  median impression ratio: 1.06x

  Could not reproduce the DIRECTION of the published finding on this slice.
  NOT evidence the paper is wrong: different dataset, and the decisive detail is
  population - that paper's un-refreshed baseline sits at 71 median impressions,
  and my filter removes 11,248 pages whose median is 31.
  Honest conclusion, narrow: refresh recency is not a priority signal HERE.


## 6. Ranked recommendations

Five archetypes carry an action. The withdrawn one is shown as withdrawn.

In [10]:
NAMES = {0:"Long-Tenured Plateau", 1:"Long-Form Untouched", 2:"High-Yield Core",
         3:"Intermittent Exposure", 4:"Recently Published, Establishing",
         5:"Measured-Engagement Minority (WITHDRAWN)"}
ACTIONS = {0:"rewrite", 1:"improve", 2:"protect", 3:"monitor", 4:"monitor", 5:"monitor"}
CONFIDENCE_FLOOR = 0.05

d["archetype"] = d["cluster"].map(NAMES)
d["action"] = d["cluster"].map(ACTIONS)
d.loc[d["silhouette"] < CONFIDENCE_FLOOR, "action"] = "monitor"
d["confidence"] = np.where((d["silhouette"] < CONFIDENCE_FLOOR) | (d["cluster"] == 5),
                           "low", "standard")

summary = d.groupby("archetype").agg(
    pages=("content_id","size"),
    median_impressions=("impressions_90d","median"), median_ctr=("ctr","median"),
    median_position=("avg_position","median"), median_age=("content_age_days","median"),
    top_client_share=("client_id", lambda s: s.value_counts().iloc[0]/len(s)),
).sort_values("pages", ascending=False)
summary["action"] = [ACTIONS[[k for k,v in NAMES.items() if v==a][0]] for a in summary.index]
print("THE ARCHETYPE -> ACTION MAPPING\n")
print(summary.round(2).to_string())
print(f"\nlow-confidence pages routed to monitor regardless of archetype: "
      f"{int((d['confidence']=='low').sum()):,} ({(d['confidence']=='low').mean()*100:.1f}%)")
print(f"distinct actions across {len(NAMES)-1} active archetypes: "
      f"{len(set(v for k,v in ACTIONS.items() if k!=5))}")
print("  -> by the falsifiable test set in ML-03, the clustering does 4 groups' worth of")
print("     decision work, not 6. Reported rather than papered over.")
print(f"\nprune / merge recommendations issued anywhere in this project: 0")

THE ARCHETYPE -> ACTION MAPPING

                                          pages  median_impressions  median_ctr  median_position  median_age  top_client_share   action
archetype                                                                                                                              
High-Yield Core                            4924             13478.0        0.36              7.4       229.0              0.31  protect
Long-Form Untouched                        4598              1650.5        0.09             15.1       263.0              0.38  improve
Recently Published, Establishing           4113              1617.0        0.16             11.7       119.0              0.14  monitor
Long-Tenured Plateau                       3595              1488.0        0.10             15.3       445.0              0.53  rewrite
Intermittent Exposure                       985               487.0        0.00             12.1       165.0              0.15  monitor
Measured-Engage

### What must never be automated

1. **Deleting, unpublishing or redirecting any page** — cluster membership describes 90 days of behaviour.
2. **Rewriting content without a human reading it** — the model has never seen the words on the page.
3. **Acting on a low-confidence row** — opposite actions have been separated by 0.1% of distance.
4. **Applying the archetypes to an unchecked new client** — leave-one-client-out ranged 0.465–1.000.
5. **Publishing per-page rows externally** — aggregates are public-safe; rows are not.

**Meta-rule:** this allocates *attention*, which is cheap to reallocate. Anything expensive to undo needs a
different input.

## 7. Artifacts the paper embeds

In [11]:
OUT, FIG = Path("work/outputs"), Path("work/figures")
DOCS = Path("docs"); SUB = Path("submission")
PAPER_URL = "https://zaindev04.github.io/Flyrank-ML-Internship/"

print("RECEIPTS - every paper number traces to one of these:\n")
for p in sorted(OUT.glob("*.json")):
    m = json.loads(p.read_text())
    print(f"  {str(p):44s} {m.get('notebook','-')}")
print()
print("FIGURES the paper embeds:")
for p in sorted(FIG.glob("*.png")) + sorted(OUT.glob("*.png")):
    print(f"  {str(p):52s} {p.stat().st_size:>8,} bytes")
print()
print("DEPLOYMENT:")
for p, what in [(DOCS/"index.html", "the paper (self-contained, images inlined)"),
                (DOCS/".nojekyll", "tells GitHub Pages to serve files as-is"),
                (SUB/"paper_url.txt", "the deployed URL, one line")]:
    ok = "OK " if p.exists() else "-- "
    extra = f"({p.stat().st_size:,} bytes)" if p.exists() else "(not in this checkout)"
    print(f"  {ok}{str(p):26s} {what} {extra}")
if (SUB/"paper_url.txt").exists():
    print(f"\n  paper_url.txt contains: {(SUB/'paper_url.txt').read_text().strip()}")

print(f"\nPAPER: {PAPER_URL}")
print(f"REPO : {REPO_URL}")

RECEIPTS - every paper number traces to one of these:

  work/outputs/baseline_metrics.json           w04_baseline_score.ipynb
  work/outputs/model_metrics.json              w05_model.ipynb
  work/outputs/playbook_metrics.json           w07_action_playbook.ipynb
  work/outputs/validation_metrics.json         w06_validation_audit.ipynb

FIGURES the paper embeds:
  work/figures/playbook_archetype_actions.png            50,316 bytes
  work/figures/playbook_confidence_and_refresh.png       69,643 bytes
  work/outputs/archetypes_vs_ladder.png                 184,474 bytes
  work/outputs/baseline_archetypes.png                   59,889 bytes
  work/outputs/validation_split_before_after.png         49,115 bytes

DEPLOYMENT:
  OK docs/index.html            the paper (self-contained, images inlined) (500,830 bytes)
  OK docs/.nojekyll             tells GitHub Pages to serve files as-is (0 bytes)
  OK submission/paper_url.txt   the deployed URL, one line (51 bytes)

  paper_url.txt contains: htt

In [12]:
# Final public-safety pass over everything this project publishes.
import re
BAD_PATTERNS = {
    "possible URL/domain": r"https?://(?!flyrank\.ai|github\.com/ZainDev04|colab\.research|huggingface\.co|zaindev04\.github\.io)[\w.-]+\.[a-z]{2,}",
    "possible client name": r"\b(?:Inc|Ltd|LLC|GmbH|Corp)\b",
    "hf token": r"hf_[A-Za-z0-9]{20,}",
}
targets = [Path("docs/index.html")] if Path("docs/index.html").exists() else []
findings = []
for t in targets:
    txt = t.read_text(encoding="utf-8", errors="ignore")
    for label, pat in BAD_PATTERNS.items():
        hits = set(re.findall(pat, txt))
        if hits: findings.append((str(t), label, list(hits)[:5]))
print("FINAL PUBLIC-SAFETY PASS\n")
if targets:
    print(f"  scanned: {[str(t) for t in targets]}")
    print(f"  findings: {findings if findings else 'none - clean'}")
else:
    print("  docs/index.html not present in this checkout (it is committed in the repo).")
print(f"\n  identifiers used throughout: pseudonyms only, e.g. "
      f"{d['content_id'].iloc[0]} | {d['client_id'].iloc[0]}")
print("  no client names, domains, URLs, page titles or raw search queries exist in the")
print("  release, the notebooks, the exports or the paper.")

FINAL PUBLIC-SAFETY PASS

  scanned: ['docs/index.html']
  findings: none - clean

  identifiers used throughout: pseudonyms only, e.g. content_304f48230142 | client_f369cb89fc
  no client names, domains, URLs, page titles or raw search queries exist in the
  release, the notebooks, the exports or the paper.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — verified in the cell above
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Capstone checklist

| Required | Where |
|---|---|
| Title + Abstract (question → method → result) | paper §Abstract |
| Introduction / problem statement | paper §1 · notebook §1 |
| Data — release, tables, windows, exclusions, public-safe | paper §2 · notebook §2 |
| Methodology — assumptions, features, label, baseline, validation, leakage | paper §3 · notebook §3 |
| Results — vs baseline on the same data, with charts | paper §4 · notebook §4 |
| Limitations & honest framing | paper §5 · notebook §5 |
| Ranked recommendations | paper §6 · notebook §6 |
| Reproducibility — notebooks and repo | paper §7 · notebook §7 |
| Acknowledgments & data credit (flyrank.ai) | paper §8 |
| Deployed at a public URL | `docs/index.html` via GitHub Pages |
| Exact URL in `submission/paper_url.txt` | one line, nothing else |

### The three sentences this work stands on

1. **Observed** — on 18,752 pages from 29 pseudonymized clients, K-Means at k = 6 produced six groups
   reproducing across random seeds at ARI 0.998.
2. **Measured** — five of the six survive removing a single feature; leave-one-client-out across 17 clients
   gives mean ARI 0.909 with three clients below 0.80 and a worst case of 0.465.
3. **Decision-support** — the output is a ranked review queue with reason codes, an explicit
   low-confidence bucket, and no automated action of any kind.

**Built on the FlyRank ML Internship dataset — [flyrank.ai](https://flyrank.ai).**